# Structure Data Manager

Notebook điều phối crawler vnstock (module `structure_data`): retry/fallback đa nguồn, QC OHLCV, pipeline một lệnh tùy chọn.

**Đồ án:** Hệ thống Data Pipeline và tra cứu phân tích thị trường chứng khoán Việt Nam đa nguồn.
**Nguồn:** `primary_source` / `fallback_source` (vd. **VCI** + **KBS**). Key vnstock: tạo file **`stock-pipeline/.env`** với dòng `VNSTOCK_API_KEY=...` (đã có trong `.gitignore`); `register_vnstock_api_key_from_env` tự gọi `load_dotenv`. Có thể dùng `.env.example` làm mẫu.

**Mục đích:** 
- **Giá OHLCV (`price`)**, **chỉ số (`index`)**, **financial ratio (`financial_ratio`)**: lưu theo **`date=<ngày chạy notebook>`**. Mỗi mã (hoặc mỗi mã chỉ số) một file Parquet trong partition đó. **Chưa có** partition/file → tạo mới; **đã có** → **ghi đè**.
- **Company overview**: file master **`company/master/company_overview.parquet`** — cơ chế **append** (mỗi lần chạy thêm batch, có `ingest_run_id`), **không** xóa lịch sử các lần trước.
- **Listing**: snapshot master tại `listing/master/` — mỗi lần chạy **ghi đè** file listing hiện tại.

**Gợi ý:** chạy từng bước (ô dưới) hoặc `run_structure_ingestion_pipeline(cfg)` để nghỉ `delay_between_categories_sec` giữa các nhóm API.

In [1]:
import os
import sys
import warnings
import threading

os.environ["PYTHONUTF8"] = "1"
os.environ["PYTHONIOENCODING"] = "utf-8"

if hasattr(sys.stdout, "reconfigure"):
    sys.stdout.reconfigure(encoding="utf-8", errors="replace")
if hasattr(sys.stderr, "reconfigure"):
    sys.stderr.reconfigure(encoding="utf-8", errors="replace")

# Suppress encoding errors từ subprocess threads
original_hook = threading.excepthook
def custom_hook(args):
    if "UnicodeDecodeError" in str(args.exc_type):
        pass  # Ignore encoding errors from threads
    else:
        original_hook(args)
threading.excepthook = custom_hook

# Suppress non-critical warnings
warnings.filterwarnings("ignore")

print("[OK] UTF-8 guard enabled for current kernel session")

[OK] UTF-8 guard enabled for current kernel session


In [ ]:
import os
import importlib
from datetime import datetime
import structure_data as sd
import structure_data.config as sd_config
import structure_data.price_ingestor as sd_price_ingestor
import structure_data.index_ingestor as sd_index_ingestor
import structure_data.stock_info_ingestor as sd_stock_info_ingestor
import structure_data.pipeline as sd_pipeline

# Reload theo thu tu: submodule -> package facade (tranh cache module cu trong notebook kernel)
for _m in (
    sd_config,
    sd_price_ingestor,
    sd_index_ingestor,
    sd_stock_info_ingestor,
    sd_pipeline,
):
    importlib.reload(_m)
sd = importlib.reload(sd)

from structure_data import (
    IngestionConfig,
    configure_logging,
    register_vnstock_api_key_from_env,
    run_structure_ingestion_pipeline,
    ingest_prices,
    ingest_indices,
    ingest_listing,
    ingest_company_overview,
    ingest_financial_ratio,
    ingest_price_board,
)

register_vnstock_api_key_from_env("VNSTOCK_API_KEY")

configure_logging()
cfg = IngestionConfig()

# Backward-compat: neu kernel dang giu dataclass cu thi bo sung field incremental
if not hasattr(cfg, "min_ohlcv_rows_stock_incremental"):
    cfg.min_ohlcv_rows_stock_incremental = 5
if not hasattr(cfg, "min_ohlcv_rows_index_incremental"):
    cfg.min_ohlcv_rows_index_incremental = 5

# TEST trong Jupyter: moi lan run tao 1 partition raw rieng (mo phong 1 DAG run)
cfg.run_partition = datetime.now().strftime("%Y-%m-%dT%H%M%S")

# Quan ly danh sach ma co phieu ngay tai notebook (them/bot moi lan chay)
cfg.tickers = [
    # VN30 hien tai (20 ma)
    "ACB","BCM","BID","BVH","CTG","FPT","GAS","GVR","HDB","HPG",
    "MBB","MSN","MWG","PLX","POW","SAB","SHB","SSI","STB","TCB",
    # Them 30 ma da nganh
    "VCB","VHM","VIC","VJC","VNM","VPB","VRE","TPB","VIB","HVN",
    "REE","PNJ","GMD","DGC","DPM","DCM","HSG","KDC","QNS","SCS",
    "NVL","PDR","DXG","KDH","HDG","EVF","KBC","AGG","TCH","VPI",
]

# Co the sua bo chi so neu can
cfg.index_tickers = ["VNINDEX", "VN30", "HNXINDEX", "HNX30", "UPCOMINDEX"]

cfg.max_tickers_per_run = len(cfg.tickers)
cfg.rate_limit_rpm = 10
cfg.inter_request_delay_sec = 0.5

# cfg.primary_source = "vci"
# cfg.fallback_source = "kbs"

RUN_PROFILE = "daily_incremental"  # "backfill" | "daily_incremental"

PROFILE_OVERRIDES = {
    "backfill": {
        "use_incremental_window": False,
        "incremental_window_days": 10,
        "bootstrap_full_history_if_missing": True,
        "full_bootstrap_once_then_incremental": False,
        "full_bootstrap_marker_file": "_full_bootstrap_done.json",
        "years_back": 5,
    },
    "daily_incremental": {
        "use_incremental_window": True,
        "incremental_window_days": 1,
        "bootstrap_full_history_if_missing": True,
        "full_bootstrap_once_then_incremental": False,
        "full_bootstrap_marker_file": "_full_bootstrap_done.json",
        "years_back": 5,
    },
}

if RUN_PROFILE not in PROFILE_OVERRIDES:
    raise ValueError(
        f"RUN_PROFILE='{RUN_PROFILE}' khong hop le. Chon 1 trong: {list(PROFILE_OVERRIDES)}"
    )

for _key, _value in PROFILE_OVERRIDES[RUN_PROFILE].items():
    setattr(cfg, _key, _value)

print("structure_data module:", sd.__file__)
print("config module:", sd_config.__file__)
print("price_ingestor module:", sd_price_ingestor.__file__)
print('has ingest_price_board:', hasattr(sd, 'ingest_price_board'))
print("RUN_PROFILE:", RUN_PROFILE)
print("fetch_mode:", "incremental-window" if cfg.use_incremental_window else "full-backfill")
print("incremental_window_days:", cfg.incremental_window_days)
print("qc stock full/inc:", cfg.min_ohlcv_rows_stock, cfg.min_ohlcv_rows_stock_incremental)
print("qc index full/inc:", cfg.min_ohlcv_rows_index, cfg.min_ohlcv_rows_index_incremental)
print("full_bootstrap_once_then_incremental:", cfg.full_bootstrap_once_then_incremental)
print('run_partition:', cfg.run_partition)
print(cfg)

📦 **Vnstock 4.0.2 is available**
Current: 3.5.1 (Python 3.12 (venv))
Update: `C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\venv\Scripts\python.exe -m pip install vnstock --upgrade`
Release: https://vnstocks.com/docs/tai-lieu/lich-su-phien-ban

📦 **Vnai 2.4.8 is available**
Current: 2.4.6 (Python 3.12 (venv))
Update: `C:\Users\ACER\Downloads\Đồ Án 2\stock-pipeline\venv\Scripts\python.exe -m pip install vnai --upgrade`
Release: https://pypi.org/project/vnai/#history

✓ API key đã được lưu thành công! (API key saved successfully!)
Bạn đang sử dụng Phiên bản cộng đồng (60 requests/phút)
(You are using Community version - 60 requests/minute)

Để tham gia gói thành viên tài trợ (To join sponsor membership):
  Truy cập: https://vnstocks.com/insiders-program
✓ API key đã được lưu thành công! vnst***6437
✓ Bạn đang sử dụng Phiên bản cộng đồng (60 requests/phút)
structure_data module: d:\WorkSpace\Đồ Án 2\stock-pipeline\ingestion\structure_data\__init__.py
config module: d:\WorkSpace\Đồ Án 2\stock-pipeline\ingestion\structure_data\config.py
price_ingestor module: d:\WorkSpace\Đồ Án 2\stock-pipeline\ingestion\structure_data\price_ingestor.py
has ingest_price_board: True
RUN_PROFILE: backfill
fetch_mode: full-backfill
incremental_window_days: 10
qc stock full/inc: 50 5
qc index full/inc: 100 5
full_bootstrap_once_then_incremental: False
run_partition: 2026-05-15T005707
IngestionConfig(tickers=['ACB', 'BCM', 'BID', 'BVH', 'CTG', 'FPT', 'GAS', 'GVR', 'HDB', 'H

In [3]:
# 1) OHLCV gia co phieu
price_outputs = ingest_prices(cfg)
len(price_outputs), list(price_outputs.items())[:3]

2026-05-15 00:57:07 [INFO] [1/50] Fetching ACB (full_5y: 2021-05-15 -> 2026-05-15, qc_min_rows=50)...


2026-05-15 00:57:10 [INFO] ACB — cột output: ['ticker', 'trading_date', 'open', 'high', 'low', 'close', 'volume', 'value', 'value_is_derived', 'source', 'instrument_type', 'ingested_at', 'fetched_at', 'is_suspicious']
2026-05-15 00:57:10 [INFO] ACB | src_used=KBS rows=1247 min_date=2021-05-17 max_date=2026-05-14 pct_missing_close=0.00% pct_missing_value=0.00%
2026-05-15 00:57:10 [INFO] Saved 1247 rows to D:\WorkSpace\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\price\date=2026-05-15T005707\ACB.parquet
2026-05-15 00:57:10 [INFO] [2/50] Fetching BCM (full_5y: 2021-05-15 -> 2026-05-15, qc_min_rows=50)...
2026-05-15 00:57:14 [INFO] BCM — cột output: ['ticker', 'trading_date', 'open', 'high', 'low', 'close', 'volume', 'value', 'value_is_derived', 'source', 'instrument_type', 'ingested_at', 'fetched_at', 'is_suspicious']
2026-05-15 00:57:14 [INFO] BCM | src_used=KBS rows=1247 min_date=2021-05-17 max_date=2026-05-14 pct_missing_close=0.00% pct_missing_value=0.00%
2026-05-15 00:57:14 [I

(50,
 [('ACB',
   'D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\price\\date=2026-05-15T005707\\ACB.parquet'),
  ('BCM',
   'D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\price\\date=2026-05-15T005707\\BCM.parquet'),
  ('BID',
   'D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\price\\date=2026-05-15T005707\\BID.parquet')])

In [4]:
# 2) Chi so VNINDEX/VN30/HNX...
index_outputs = ingest_indices(cfg)
index_outputs

2026-05-15 01:02:02 [INFO] [1/5] Fetching index VNINDEX (full_5y: 2021-05-15 -> 2026-05-15, qc_min_rows=100)...
2026-05-15 01:02:08 [INFO] VNINDEX — cột output: ['ticker', 'trading_date', 'open', 'high', 'low', 'close', 'volume', 'value', 'value_is_derived', 'source', 'instrument_type', 'ingested_at', 'fetched_at', 'is_suspicious']
2026-05-15 01:02:08 [INFO] VNINDEX | src_used=KBS rows=1247 min_date=2021-05-17 max_date=2026-05-14 pct_missing_close=0.00% pct_missing_value=0.00%
2026-05-15 01:02:08 [INFO] Saved 1247 rows to D:\WorkSpace\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\index\date=2026-05-15T005707\VNINDEX.parquet
2026-05-15 01:02:08 [INFO] [2/5] Fetching index VN30 (full_5y: 2021-05-15 -> 2026-05-15, qc_min_rows=100)...
2026-05-15 01:02:14 [INFO] VN30 — cột output: ['ticker', 'trading_date', 'open', 'high', 'low', 'close', 'volume', 'value', 'value_is_derived', 'source', 'instrument_type', 'ingested_at', 'fetched_at', 'is_suspicious']
2026-05-15 01:02:14 [INFO] VN30 | 

{'VNINDEX': 'D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\index\\date=2026-05-15T005707\\VNINDEX.parquet',
 'VN30': 'D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\index\\date=2026-05-15T005707\\VN30.parquet',
 'HNXINDEX': 'D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\index\\date=2026-05-15T005707\\HNXINDEX.parquet',
 'HNX30': 'D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\index\\date=2026-05-15T005707\\HNX30.parquet',
 'UPCOMINDEX': 'D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\index\\date=2026-05-15T005707\\UPCOMINDEX.parquet'}

In [5]:
# 3) Listing
listing_file = ingest_listing(cfg)
listing_file

2026-05-15 01:02:38 [INFO] symbols_by_exchange: 3255 mã (có sàn) từ KBS
2026-05-15 01:02:52 [INFO] Lọc stock: 3255 -> 1536 dòng
2026-05-15 01:02:52 [INFO] ✓ Đã ghi 1536 dòng -> D:\WorkSpace\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\listing\master\listing.parquet


'D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\listing\\master\\listing.parquet'

In [6]:
# 4) Company overview (master append)
company_file = ingest_company_overview(cfg)
company_file

2026-05-15 01:02:52 [INFO] [1/50] Đang tải company ACB...
2026-05-15 01:02:53 [INFO] [2/50] Đang tải company BCM...
2026-05-15 01:02:59 [INFO] [3/50] Đang tải company BID...
2026-05-15 01:03:05 [INFO] [4/50] Đang tải company BVH...
2026-05-15 01:03:11 [INFO] [5/50] Đang tải company CTG...
2026-05-15 01:03:17 [INFO] [6/50] Đang tải company FPT...
2026-05-15 01:03:23 [INFO] [7/50] Đang tải company GAS...
2026-05-15 01:03:29 [INFO] [8/50] Đang tải company GVR...
2026-05-15 01:03:35 [INFO] [9/50] Đang tải company HDB...
2026-05-15 01:03:41 [INFO] [10/50] Đang tải company HPG...
2026-05-15 01:03:47 [INFO] [11/50] Đang tải company MBB...
2026-05-15 01:03:53 [INFO] [12/50] Đang tải company MSN...
2026-05-15 01:03:59 [INFO] [13/50] Đang tải company MWG...
2026-05-15 01:04:05 [INFO] [14/50] Đang tải company PLX...
2026-05-15 01:04:11 [INFO] [15/50] Đang tải company POW...
2026-05-15 01:04:17 [INFO] [16/50] Đang tải company SAB...
2026-05-15 01:04:23 [INFO] [17/50] Đang tải company SHB...
2026-0

'D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\company\\master\\company_overview.parquet'

In [7]:
# 5) Financial ratio
ratio_outputs = ingest_financial_ratio(cfg)
len(ratio_outputs), list(ratio_outputs.items())[:3]

2026-05-15 01:07:47 [INFO] [1/50] Đang tải financial ratio ACB...
2026-05-15 01:07:52 [INFO] Loaded 162 built-in KBS field mappings
2026-05-15 01:07:52 [INFO] Loaded 162 built-in KBS field mappings
2026-05-15 01:07:57 [WARNING] ratio ACB@kbs/quarter attempt 1/2 failed (RetryError), retry in 1.21s
2026-05-15 01:07:58 [INFO] Loaded 162 built-in KBS field mappings
2026-05-15 01:07:58 [INFO] Loaded 162 built-in KBS field mappings
2026-05-15 01:08:03 [WARNING] Finance ratio ACB (kbs, quarter): RetryError[<Future at 0x20b21ede5a0 state=finished raised ConnectionError>]
2026-05-15 01:08:03 [WARNING] Finance source kbs tạm disable cho phần còn lại của run (lỗi kết nối/retry lặp lại).
2026-05-15 01:08:05 [WARNING] Finance ratio ACB (vci, quarter): 'data'
2026-05-15 01:08:05 [WARNING] Finance source vci tạm disable cho phần còn lại của run (lỗi không tương thích response).
2026-05-15 01:08:05 [WARNING] Bỏ qua ACB — financial_ratio rỗng mọi nguồn/kỳ
2026-05-15 01:08:05 [WARNING] Dừng financial_ra

(0, [])

In [8]:
# 6) Price board snapshot
price_board_file = ingest_price_board(cfg)
price_board_file

2026-05-15 01:08:11 [INFO] Saved 50 rows to D:\WorkSpace\Đồ Án 2\stock-pipeline\data-lake\raw\Structure_Data\price_board\date=2026-05-15T005707\PRICE_BOARD_SNAPSHOT.parquet


'D:\\WorkSpace\\Đồ Án 2\\stock-pipeline\\data-lake\\raw\\Structure_Data\\price_board\\date=2026-05-15T005707\\PRICE_BOARD_SNAPSHOT.parquet'

In [9]:
# 7) Tong ket structure-data (gon + day du)
from pathlib import Path
from datetime import datetime
import pandas as pd

base = Path(cfg.data_lake_root)

PARTITIONED_CATEGORIES = ["price", "index", "financial_ratio", "price_board"]
EXPECTED_PARTITIONED = set(PARTITIONED_CATEGORIES)
MASTER_CHECKS = {
    "listing": base / "listing" / "master" / "listing.parquet",
    "company": base / "company" / "master" / "company_overview.parquet",
}

def _parquet_rows(file_path: Path) -> int:
    try:
        import pyarrow.parquet as pq
        return int(pq.ParquetFile(file_path).metadata.num_rows)
    except Exception:
        try:
            return int(pd.read_parquet(file_path).shape[0])
        except Exception:
            return -1

def _human_size(num_bytes: int) -> str:
    units = ["B", "KB", "MB", "GB", "TB"]
    value = float(max(int(num_bytes), 0))
    for unit in units:
        if value < 1024 or unit == units[-1]:
            return f"{value:,.2f} {unit}"
        value /= 1024
    return f"{num_bytes} B"

def _sum_rows_with_unknown(series: pd.Series) -> int:
    values = pd.to_numeric(series, errors="coerce").fillna(-1).astype(int)
    return -1 if (values < 0).any() else int(values.sum())

def _partition_datetime(label: str) -> pd.Timestamp:
    txt = str(label).strip()
    if "T" in txt:
        dt = pd.to_datetime(txt, format="%Y-%m-%dT%H%M%S", errors="coerce")
        if pd.notna(dt):
            return dt
    return pd.to_datetime(txt, errors="coerce")

def _split_partition(label: str) -> tuple[str, str, str]:
    dt = _partition_datetime(label)
    if pd.isna(dt):
        return "UNKNOWN", "", str(label)
    return dt.strftime("%Y-%m-%d"), dt.strftime("%H:%M:%S"), str(label)

partition_rows = []
for category in PARTITIONED_CATEGORIES:
    category_root = base / category
    if not category_root.exists():
        continue

    date_dirs = sorted(p for p in category_root.glob("date=*") if p.is_dir())
    for day_dir in date_dirs:
        raw_partition = day_dir.name.split("=", 1)[-1]
        run_date, run_time, partition_label = _split_partition(raw_partition)
        files = sorted(day_dir.glob("*.parquet"))

        total_size_bytes = 0
        total_rows = 0
        row_error = False
        largest_file = ""
        largest_file_size_bytes = 0

        for f in files:
            try:
                file_size = int(f.stat().st_size)
            except Exception:
                file_size = 0
            total_size_bytes += file_size

            rows = _parquet_rows(f)
            if rows >= 0:
                total_rows += rows
            else:
                row_error = True

            if file_size > largest_file_size_bytes:
                largest_file_size_bytes = file_size
                largest_file = f.name

        partition_rows.append({
            "partition": partition_label,
            "run_date": run_date,
            "run_time": run_time,
            "category": category,
            "status": "OK" if files else "MISSING",
            "file_count": len(files),
            "total_rows": total_rows if files and not row_error else (0 if not files else -1),
            "total_size_bytes": total_size_bytes,
            "total_size_human": _human_size(total_size_bytes),
            "avg_file_size_mb": round(total_size_bytes / len(files) / (1024 ** 2), 3) if files else 0,
            "largest_file": largest_file,
            "largest_file_size_human": _human_size(largest_file_size_bytes),
        })

partition_df = pd.DataFrame(partition_rows)
partition_total_bytes = 0

if partition_df.empty:
    print("Khong tim thay partition date=* trong structure-data.")
else:
    partition_df["dt_sort"] = partition_df["partition"].map(_partition_datetime)
    partition_df = partition_df.sort_values(
        by=["dt_sort", "category"],
        ascending=[False, True],
        kind="stable",
    ).reset_index(drop=True)
    partition_df["total_size_mb"] = (partition_df["total_size_bytes"] / (1024 ** 2)).round(3)
    partition_df["total_size_gb"] = (partition_df["total_size_bytes"] / (1024 ** 3)).round(4)

    print("A) Chi tiet theo run partition + category")
    display(
        partition_df[[
            "partition",
            "run_date",
            "run_time",
            "category",
            "status",
            "file_count",
            "total_rows",
            "total_size_human",
            "total_size_mb",
            "avg_file_size_mb",
            "largest_file",
            "largest_file_size_human",
        ]]
    )

    categories_by_day = (
        partition_df.groupby("run_date")["category"]
        .agg(lambda s: sorted(set(s)))
        .reset_index(name="categories")
    )

    daily_df = partition_df.groupby("run_date", as_index=False).agg(
        total_files=("file_count", "sum"),
        total_rows=("total_rows", _sum_rows_with_unknown),
        total_size_bytes=("total_size_bytes", "sum"),
    )
    daily_df = daily_df.merge(categories_by_day, on="run_date", how="left")
    daily_df["categories_present"] = daily_df["categories"].apply(
        lambda xs: len(xs) if isinstance(xs, list) else 0
    )
    daily_df["categories"] = daily_df["categories"].apply(
        lambda xs: ", ".join(xs) if isinstance(xs, list) else ""
    )
    daily_df["missing_categories"] = daily_df["categories"].apply(
        lambda txt: ", ".join(sorted(EXPECTED_PARTITIONED - set([x.strip() for x in txt.split(",") if x.strip()])))
    )
    daily_df["coverage_ok"] = daily_df["missing_categories"].eq("")
    daily_df["total_size_mb"] = (daily_df["total_size_bytes"] / (1024 ** 2)).round(3)
    daily_df["total_size_gb"] = (daily_df["total_size_bytes"] / (1024 ** 3)).round(4)
    daily_df["total_size_human"] = daily_df["total_size_bytes"].map(_human_size)
    daily_df["run_date_sort"] = pd.to_datetime(daily_df["run_date"], errors="coerce")
    daily_df = daily_df.sort_values(by="run_date_sort", ascending=False, kind="stable").reset_index(drop=True)

    print("B) Tong theo ngay + coverage category")
    display(
        daily_df[[
            "run_date",
            "categories_present",
            "categories",
            "missing_categories",
            "coverage_ok",
            "total_files",
            "total_rows",
            "total_size_human",
            "total_size_mb",
            "total_size_gb",
        ]]
    )

    latest_by_category = (
        partition_df.sort_values(by="dt_sort", ascending=False, kind="stable")
        .drop_duplicates(subset=["category"], keep="first")[
            ["category", "partition", "run_date", "run_time"]
        ]
        .rename(
            columns={
                "partition": "latest_partition",
                "run_date": "latest_run_date",
                "run_time": "latest_run_time",
            }
        )
    )

    category_total_df = partition_df.groupby("category", as_index=False).agg(
        crawl_days=("run_date", "nunique"),
        total_files=("file_count", "sum"),
        total_rows=("total_rows", _sum_rows_with_unknown),
        total_size_bytes=("total_size_bytes", "sum"),
    )
    category_total_df["total_size_mb"] = (category_total_df["total_size_bytes"] / (1024 ** 2)).round(3)
    category_total_df["total_size_gb"] = (category_total_df["total_size_bytes"] / (1024 ** 3)).round(4)
    category_total_df["total_size_human"] = category_total_df["total_size_bytes"].map(_human_size)
    category_total_df = category_total_df.merge(latest_by_category, on="category", how="left")
    category_total_df = category_total_df.sort_values(
        by=["total_size_bytes", "total_rows"],
        ascending=[False, False],
        kind="stable",
    ).reset_index(drop=True)

    print("C) Luy ke theo category + latest run")
    display(
        category_total_df[[
            "category",
            "crawl_days",
            "total_files",
            "total_rows",
            "total_size_human",
            "total_size_mb",
            "total_size_gb",
            "latest_partition",
            "latest_run_date",
            "latest_run_time",
        ]]
    )

    partition_total_bytes = int(partition_df["total_size_bytes"].sum())

master_rows = []
for category, file_path in MASTER_CHECKS.items():
    exists = file_path.exists()
    size_bytes = 0
    rows = 0
    last_modified = ""

    if exists:
        try:
            stat = file_path.stat()
            size_bytes = int(stat.st_size)
            last_modified = datetime.fromtimestamp(stat.st_mtime).strftime("%Y-%m-%d %H:%M:%S")
        except Exception:
            size_bytes = 0
        rows = _parquet_rows(file_path)

    master_rows.append({
        "category": category,
        "status": "OK" if exists else "MISSING",
        "file": file_path.name if exists else "",
        "rows": rows if exists else 0,
        "size_bytes": size_bytes,
        "size_human": _human_size(size_bytes),
        "last_modified": last_modified,
    })

master_df = pd.DataFrame(master_rows)
master_total_bytes = int(master_df["size_bytes"].sum()) if not master_df.empty else 0

print("D) Master snapshot (khong partition theo ngay)")
display(master_df)

overall_total_bytes = partition_total_bytes + master_total_bytes
print("E) Tong dung luong")
print(f"- Partition (tat ca run): {_human_size(partition_total_bytes)}")
print(f"- Master snapshot: {_human_size(master_total_bytes)}")
print(f"- Structure-data tong: {_human_size(overall_total_bytes)}")

if not partition_df.empty:
    latest_dt = partition_df["dt_sort"].dropna().max()
    if pd.notna(latest_dt):
        print(f"Latest run partition: {latest_dt.strftime('%Y-%m-%d %H:%M:%S')}")

A) Chi tiet theo run partition + category


,partition,run_date,run_time,category,status,file_count,total_rows,total_size_human,total_size_mb,avg_file_size_mb,largest_file,largest_file_size_human
0,2026-05-15T005707,2026-05-15,00:57:07,index,OK,5,6234,190.59 KB,0.186,0.037,VNINDEX.parquet,40.66 KB
1,2026-05-15T005707,2026-05-15,00:57:07,price,OK,50,62339,2.38 MB,2.375,0.048,DGC.parquet,54.66 KB
2,2026-05-15T005707,2026-05-15,00:57:07,price_board,OK,1,50,29.29 KB,0.029,0.029,PRICE_BOARD_SNAPSHOT.parquet,29.29 KB
3,2026-05-08T132323,2026-05-08,13:23:23,index,OK,5,6234,190.70 KB,0.186,0.037,VNINDEX.parquet,40.66 KB
4,2026-05-08T132323,2026-05-08,13:23:23,price,OK,50,62339,2.38 MB,2.377,0.048,DGC.parquet,54.70 KB
5,2026-05-08T132323,2026-05-08,13:23:23,price_board,OK,1,50,29.20 KB,0.029,0.029,PRICE_BOARD_SNAPSHOT.parquet,29.20 KB
6,2026-04-24T134913,2026-04-24,13:49:13,index,OK,5,6238,190.56 KB,0.186,0.037,VNINDEX.parquet,40.62 KB
7,2026-04-24T134913,2026-04-24,13:49:13,price,OK,50,62389,2.38 MB,2.379,0.048,DGC.parquet,54.78 KB
8,2026-04-24T134913,2026-04-24,13:49:13,price_board,OK,1,50,29.32 KB,0.029,0.029,PRICE_BOARD_SNAPSHOT.parquet,29.32 KB
9,2026-04-22T001233,2026-04-22,00:12:33,index,OK,5,6237,190.48 KB,0.186,0.037,VNINDEX.parquet,40.61 KB


B) Tong theo ngay + coverage category


,run_date,categories_present,categories,missing_categories,coverage_ok,total_files,total_rows,total_size_human,total_size_mb,total_size_gb
0,2026-05-15,3,"index, price, price_board",financial_ratio,False,56,68623,2.59 MB,2.590,0.0025
1,2026-05-08,3,"index, price, price_board",financial_ratio,False,56,68623,2.59 MB,2.592,0.0025
2,2026-04-24,3,"index, price, price_board",financial_ratio,False,56,68677,2.59 MB,2.594,0.0025
3,2026-04-22,3,"index, price, price_board",financial_ratio,False,56,68639,2.59 MB,2.594,0.0025
4,2026-04-11,1,financial_ratio,"index, price, price_board",False,50,2394,2.16 MB,2.157,0.0021
5,UNKNOWN,2,"index, price","financial_ratio, price_board",False,2,8,17.51 KB,0.017,0.0000


C) Luy ke theo category + latest run


,category,crawl_days,total_files,total_rows,total_size_human,total_size_mb,total_size_gb,latest_partition,latest_run_date,latest_run_time
0,price,5,201,249423,9.52 MB,9.519,0.0093,2026-05-15T005707,2026-05-15,00:57:07
1,financial_ratio,1,50,2394,2.16 MB,2.157,0.0021,2026-04-11,2026-04-11,00:00:00
2,index,5,21,24947,771.10 KB,0.753,0.0007,2026-05-15T005707,2026-05-15,00:57:07
3,price_board,4,4,200,117.14 KB,0.114,0.0001,2026-05-15T005707,2026-05-15,00:57:07


D) Master snapshot (khong partition theo ngay)


,category,status,file,rows,size_bytes,size_human,last_modified
0,listing,OK,listing.parquet,1536,68095,66.50 KB,2026-05-15 01:02:52
1,company,OK,company_overview.parquet,200,94253,92.04 KB,2026-05-15 01:07:47


E) Tong dung luong
- Partition (tat ca run): 12.54 MB
- Master snapshot: 158.54 KB
- Structure-data tong: 12.70 MB
Latest run partition: 2026-05-15 00:57:07
